# VoiceGuard – Wav2Vec 2.0 Transformer (Model 1)
DFADD experiment for Google Colab. Select Runtime → Change runtime type → T4 GPU before running the code cell. This notebook trains a frozen Wav2Vec 2.0 encoder with a binary classification head, evaluates it, and saves the checkpoint and CSV results.

In [ ]:
!pip install -q transformers datasets librosa soundfile accelerate scikit-learn


In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import librosa

from datasets import load_dataset
from transformers import (
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Model
)
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))



Device: cpu


In [ ]:
print("\nLoading DFADD dataset...")

ds = load_dataset(
    "SpeechAntiSpoofingBenchmarks/DFADD",
    split="test"
)

real_ids = [
    i for i, row in enumerate(ds)
    if int(row["label"]) == 0
]

fake_ids = [
    i for i, row in enumerate(ds)
    if int(row["label"]) == 1
]

print("Real samples:", len(real_ids))
print("Fake samples:", len(fake_ids))




Loading DFADD dataset...


README.md:   0%|          | 0.00/3.70k [00:00<?, ?B/s]

data/test-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  302MB            

data/test-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/test-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  239MB            

data/test-00001-of-00002.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/3755 [00:00<?, ? examples/s]

Real samples: 755
Fake samples: 3000


In [ ]:
rng = np.random.default_rng(SEED)

selected_real = rng.choice(
    real_ids, size=min(755, len(real_ids)), replace=False
)

selected_fake = rng.choice(
    fake_ids, size=min(755, len(fake_ids)), replace=False
)

selected_ids = np.concatenate([selected_real, selected_fake])
rng.shuffle(selected_ids)

In [ ]:
print("\nPreprocessing audio...")

TARGET_SR = 16000
MAX_SECONDS = 5
MAX_LENGTH = TARGET_SR * MAX_SECONDS

audio_data = []
labels = []

for idx in selected_ids:
    row = ds[int(idx)]

    audio = np.asarray(row["audio"]["array"], dtype=np.float32)
    sr = row["audio"]["sampling_rate"]

    # Convert stereo to mono
    if audio.ndim > 1:
        audio = np.mean(audio, axis=0 if audio.shape[0] <= 2 else -1)

    # Resample to 16 kHz
    if sr != TARGET_SR:
        audio = librosa.resample(
            audio, orig_sr=sr, target_sr=TARGET_SR
        )

    # Remove invalid values
    audio = np.nan_to_num(audio)

    # Trim long recordings
    audio = audio[:MAX_LENGTH]

    # Pad short recordings
    if len(audio) < MAX_LENGTH:
        audio = np.pad(
            audio, (0, MAX_LENGTH - len(audio))
        )

    audio_data.append(audio)
    labels.append(int(row["label"]))

audio_data = np.asarray(audio_data, dtype=np.float32)
labels = np.asarray(labels, dtype=np.int64)

print("Audio shape:", audio_data.shape)
print("Labels shape:", labels.shape)


Preprocessing audio...
Audio shape: (1510, 80000)
Labels shape: (1510,)


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    audio_data,
    labels,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("\nTrain:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))


Train: 1057
Validation: 226
Test: 227


In [ ]:
MODEL_NAME = "facebook/wav2vec2-base"

print("\nLoading pretrained Wav2Vec 2.0...")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    MODEL_NAME
)



Loading pretrained Wav2Vec 2.0...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

In [ ]:
class AudioDataset(Dataset):

    def __init__(self, audios, labels):
        self.audios = audios
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):

        audio = self.audios[index]

        inputs = feature_extractor(
            audio,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
            padding="max_length",
            max_length=MAX_LENGTH,
            truncation=True
        )

        return {
            "input_values": inputs.input_values.squeeze(0),
            "label": torch.tensor(
                self.labels[index], dtype=torch.long
            )
        }

BATCH_SIZE = 4

train_loader = DataLoader(
    AudioDataset(X_train, y_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    AudioDataset(X_val, y_val),
    batch_size=BATCH_SIZE
)

test_loader = DataLoader(
    AudioDataset(X_test, y_test),
    batch_size=BATCH_SIZE
)


In [ ]:
class VoiceGuardWav2Vec(nn.Module):

    def __init__(self):
        super().__init__()

        self.wav2vec = Wav2Vec2Model.from_pretrained(
            MODEL_NAME
        )

        # Freeze pretrained encoder initially
        for param in self.wav2vec.parameters():
            param.requires_grad = False

        hidden_size = self.wav2vec.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, input_values):

        outputs = self.wav2vec(
            input_values=input_values
        )

        # Mean pooling over time
        features = outputs.last_hidden_state.mean(dim=1)

        return self.classifier(features)


model = VoiceGuardWav2Vec().to(device)


config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  380MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  380MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.classifier.parameters(),
    lr=1e-3
)

EPOCHS = 5

In [ ]:
MAX_SECONDS = 3
MAX_LENGTH = TARGET_SR * MAX_SECONDS

# When selecting samples:
selected_real = rng.choice(real_ids, size=min(200, len(real_ids)), replace=False)
selected_fake = rng.choice(fake_ids, size=min(200, len(fake_ids)), replace=False)

EPOCHS = 2
BATCH_SIZE = 8

In [ ]:
best_val_loss = float("inf")
best_state = None
history = []

for epoch in range(EPOCHS):
    # Training
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        inputs = batch["input_values"].to(device)
        targets = batch["label"].to(device)

        optimizer.zero_grad()

        # Do not use torch.no_grad() here: the classifier needs gradients.
        logits = model(inputs)
        loss = criterion(logits, targets)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for batch in val_loader:
            inputs = batch["input_values"].to(device)
            targets = batch["label"].to(device)

            logits = model(inputs)
            loss = criterion(logits, targets)
            val_loss += loss.item()

            probabilities = torch.softmax(logits, dim=1)[:, 1]
            val_preds.extend(probabilities.cpu().numpy())
            val_labels.extend(targets.cpu().numpy())

    val_loss /= len(val_loader)

    val_classes = (np.array(val_preds) >= 0.5).astype(int)
    val_acc = accuracy_score(val_labels, val_classes)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
    })

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_acc:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

if best_state is None:
    raise RuntimeError("No best model state was saved. Check that val_loader is not empty.")

# Restore the model with the lowest validation loss
model.load_state_dict(best_state)

In [ ]:
model.eval()

test_probabilities = []
test_labels = []

with torch.no_grad():

    for batch in test_loader:

        inputs = batch["input_values"].to(device)
        targets = batch["label"].to(device)

        logits = model(inputs)

        probabilities = torch.softmax(logits, dim=1)[:, 1]

        test_probabilities.extend(
            probabilities.cpu().numpy()
        )

        test_labels.extend(targets.cpu().numpy())

test_probabilities = np.array(test_probabilities)
test_labels = np.array(test_labels)

test_predictions = (
    test_probabilities >= 0.5
).astype(int)

accuracy = accuracy_score(test_labels, test_predictions)
precision = precision_score(
    test_labels, test_predictions, zero_division=0
)
recall = recall_score(
    test_labels, test_predictions, zero_division=0
)
f1 = f1_score(
    test_labels, test_predictions, zero_division=0
)
roc_auc = roc_auc_score(
    test_labels, test_probabilities
)


In [ ]:
results = pd.DataFrame([{
    "Model": "Wav2Vec2",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "ROC-AUC": roc_auc
}])

print("\n========== WAV2VEC 2.0 RESULTS ==========")
print(results.to_string(index=False))

print("\nClassification Report:")
print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=["Real", "Fake"],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(test_labels, test_predictions))


In [ ]:
os.makedirs("/content/VoiceGuard_Wav2Vec2", exist_ok=True)

torch.save(
    model.state_dict(),
    "/content/VoiceGuard_Wav2Vec2/model.pth"
)

results.to_csv(
    "/content/VoiceGuard_Wav2Vec2/results.csv",
    index=False
)

pd.DataFrame(history).to_csv(
    "/content/VoiceGuard_Wav2Vec2/training_history.csv",
    index=False
)

print("\nModel and results saved successfully.")
print("Folder: /content/VoiceGuard_Wav2Vec2")